In [1]:
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq                          # replaces ChatOpenAI
import os

llm = ChatGroq(model="llama-3.3-70b-versatile")

In [50]:
# from langchain_community.tools import DuckDuckGoSearchRun

# search = DuckDuckGoSearchRun()

# search.invoke("what is the capital of france?")

In [51]:
from langchain.tools import tool

In [52]:
@tool
def tool_duckduckgo_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about current events or general knowledge. """

    from langchain_community.tools import DuckDuckGoSearchRun

    search = DuckDuckGoSearchRun()

    response = search.invoke(query)

    return response

In [53]:
tool_duckduckgo_search.invoke("What is the capital of France?")

'France (/ˈfræns/ ⓘ or /ˈfrɑːns/; French pronunciation: [fʁɑ̃s]), officially the French Republic (French: République française, French pronunciation: [ʁepyblik fʁɑ̃sɛz]), is a country in Western Europe. It also includes various departments and territories of France overseas. Mainland France extends from the Mediterranean Sea to the English Channel and the North Sea, and from ... Capital cities of Europe in alphabetical order, from Albania to Turkiye. Discover 49 European countries and their most important cities. France Capital: Paris Paris - The City of Light Known as "The City of Light," Paris is renowned for its rich cultural heritage, historic landmarks, and vibrant arts scene. As the capital of France, it has been a pivotal center of politics, philosophy, and fashion throughout history, shaping global culture and trends. Timeline of Paris Discover why Paris is the capital of France, its historical evolution, political role, economic impact, and cultural significance in shaping Fre

In [54]:
@tool 
def tool_wikipedia_search(query: str) -> str:
    """Use this tool when you need to answer questions about persons, places, etc."""

    import wikipedia  # ← add this
    wikipedia.set_user_agent("MyLangChainApp/1.0 (your-email@example.com)")  # ← and this

    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper

    wikipedia_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

    response = wikipedia_tool.invoke(query)

    return response

In [55]:
tool_wikipedia_search.invoke("Barak Obama")

"Page: Barack Obama\nSummary: Barack Hussein Obama II (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. A member of the Democratic Party, he was the first African American president. Obama previously served as a U.S. senator representing Illinois from 2005 to 2008 and as an Illinois state senator from 1997 to 2004.\nBorn in Honolulu, Obama graduated from Columbia University in 1983 with a Bachelor of Arts degree in political science and later worked as a community organizer in Chicago. In 1988, Obama enrolled in Harvard Law School, where he was the first Black president of the Harvard Law Review. He became a civil rights attorney and an academic, teaching constitutional law at the University of Chicago Law School from 1992 to 2004. In 1996, Obama was elected to represent the 13th district in the Illinois Senate, a position he held until 2004, when he successfully ran for the U.S. Senate. In the 2008 presidential ele

In [56]:
@tool
def tool_arxiv_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about scientific papers or research topics. """

    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    # 1. Initialize the arXiv API wrapper
    arxiv_wrapper = ArxivAPIWrapper(
        top_k_results=3,       # Number of papers to retrieve
        doc_content_chars_max=2000  # Max characters per document
    )

    # 2. Create the arXiv tool
    arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

    # 3. Use the tool directly
    result = arxiv_tool.run(query)

    print(result)

In [57]:
@tool
def tool_personal_info(name: str) -> str:
    """Use this tool when you need to answer questions about personal information.
    Args:
        name (str): The name of the person to look up.
    Returns:
        str: A string containing the person's age and occupation, or a message if the information is not found.
    """
    
    infos = [{
        "name": "Dhruv Patel",
        "age": 25,
        "occupation": "Data analyst"
    },
    {
        "name": "Jane Smith",
        "age": 25,
        "occupation": "Data Scientist"
    }]

    for info in infos:
        if info["name"].lower() == name.lower():
            return f"{info['name']} is {info['age']} years old and works as a {info['occupation']}."
    return "Information not found."

In [58]:
tool_personal_info.invoke("Dhruv Patel")

'Dhruv Patel is 25 years old and works as a Data analyst.'

## Bind Tools

In [59]:
toolkit = [
    tool_duckduckgo_search,
    tool_wikipedia_search,
    tool_arxiv_search,
    tool_personal_info
]

In [60]:
llm_bind = llm.bind_tools(toolkit)

In [61]:
llm_bind.invoke("What's the age of Dhruv Patel?. Make tool calls if necessary.")

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'zg0p8vx3w', 'function': {'arguments': '{"name":"Dhruv Patel"}', 'name': 'tool_personal_info'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 555, 'total_tokens': 575, 'completion_time': 0.066429054, 'completion_tokens_details': None, 'prompt_time': 0.054731333, 'prompt_tokens_details': None, 'queue_time': 0.047478753, 'total_time': 0.121160387}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ba38bbab80', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e1bd3-e0db-7183-9d19-b18aca8d36e2-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'Dhruv Patel'}, 'id': 'zg0p8vx3w', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 555, 'output_tokens': 20, 'total_tokens': 575})

## ReAct agent

In [62]:
from langchain.agents import create_agent


my_agent = create_agent(llm_bind, toolkit)

In [64]:
my_agent.invoke(
    {"messages": [{"role": "user", "content": "What's the age of Jane Smith?. Make tool calls if necessary."}]}
)

{'messages': [HumanMessage(content="What's the age of Jane Smith?. Make tool calls if necessary.", additional_kwargs={}, response_metadata={}, id='b52eb383-61ca-4e4d-afcc-7692592af27a'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '94vb8x219', 'function': {'arguments': '{"name":"Jane Smith"}', 'name': 'tool_personal_info'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 553, 'total_tokens': 570, 'completion_time': 0.072130008, 'completion_tokens_details': None, 'prompt_time': 0.087160811, 'prompt_tokens_details': None, 'queue_time': 0.061576358, 'total_time': 0.159290819}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ba38bbab80', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e1bd4-26f5-7193-b40d-6685ea8b96f3-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'Jane Smith'}, 'id': '94vb8x219', 'type': 'tool_